# Wstęp
Projekt powstał w oparciu o bazę danych RAVDESS Emotional Speech Audio, zawierającą nagrania wypowiedzi o różnym zabarwieniu emocjonalnym. Zbiór ten składa się z próbek mowy aktorów, w których każda wypowiedź została jednoznacznie oznaczona jedną z ośmiu klas emocji:
*(01 – neutralna, 02 – spokojna, 03 – szczęśliwa, 04 – smutna, 05 – zła, 06 – przestraszona, 07 – wstręt, 08 – zaskoczona)*.

Cała baza danych jest stosunkowo niewielka, zawiera 1440 plików audio.

Celem projektu było zaprojektowanie oraz analiza kompletnego pipeline’u klasyfikacji emocji w mowie, obejmującego ekstrakcję cech, ich przetwarzanie oraz klasyfikację przy użyciu wybranych algorytmów uczenia maszynowego. W szczególności skupiono się na porównaniu skuteczności modeli Support Vector Machine (SVM) oraz Random Forest zarówno w zadaniu klasyfikacji wieloklasowej, jak i binarnej.

W ramach pracy wykorzystano reprezentacje sygnału mowy, takie jak współczynniki MFCC wraz z pochodnymi (delta i delta-delta) oraz spektrogram, które następnie zostały opisane za pomocą zestawu statystyk. Dodatkowo przeprowadzono proces optymalizacji parametrów modeli z użyciem walidacji krzyżowej oraz narzędzia Optuna.

In [ ]:
!pip install kagglehub librosa optuna

In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay as CMD
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import sys
import os
import warnings
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.append(os.path.abspath('..'))


# imports from src
from src.data_loader import load_ravdess_dataframe, split_by_actors
from src.feature_extraction import feature_extractor
from src.train import perform_scaler, perform_pca, optimize_svm, optimize_rf


%matplotlib inline

Wczytanie bazy danych oraz jej podział.

In [ ]:
df = load_ravdess_dataframe()
print(f"Pomyślnie wczytano {len(df)} plików audio.")
print(f"Liczebność klas: {Counter(df['Emotions'])}")
train_df, test_df = split_by_actors(df)
print(f"\nPodział na zbiory (bez wycieku danych - unikalni aktorzy):")
print(f"Zbiór treningowy: {len(train_df)} próbek.")
print(f"Zbiór testowy: {len(test_df)} próbek.")

Ekstrakcja cech audio

In [ ]:
train_paths = train_df['Path'].tolist()
test_paths = test_df['Path'].tolist()

print("Rozpoczęcie ekstrakcji cech (MFCC, Spectrogram)... Proszę czekać.")
mfcc_train, spec_train = feature_extractor(train_paths)
mfcc_test, spec_test = feature_extractor(test_paths)

print(f"\nWymiary po ekstrakcji:")
print(f"MFCC: {mfcc_train.shape}")
print(f"Spektrogram: {spec_train.shape}")

Standaryzacja, PCA i łączenie cech

In [ ]:
# Standaryzacja
mfcc_train_std, mfcc_test_std = perform_scaler(mfcc_train, mfcc_test)
spec_train_std, spec_test_std = perform_scaler(spec_train, spec_test)

# Redukcja wymiarowości
spec_train_pca, spec_test_pca = perform_pca(spec_train_std, spec_test_std, n_components=100)

# Łączenie w ostateczne wektory
X_train = np.concatenate([mfcc_train_std, spec_train_pca], axis=1)
X_test  = np.concatenate([mfcc_test_std,  spec_test_pca], axis=1)
y_train = train_df['Emotions'].values
y_test  = test_df['Emotions'].values

print(f"Ostateczny wymiar wektora cech wejściowych (X_train): {X_train.shape}")

Trening i optymalizacja SVM

In [ ]:
print("Szukanie optymalnych parametrów dla SVM za pomocą Optuny (20 prób)...")
best_params_svm, best_score_svm = optimize_svm(X_train, y_train, n_trials=20)

print(f"\nNajlepsze parametry SVM: {best_params_svm}")
print(f"Najwyższy wynik CV (F1): {best_score_svm:.4f}")

# Trening ostatecznego modelu
best_model_svm = SVC(**best_params_svm)
best_preds_svm = best_model_svm.fit(X_train, y_train).predict(X_test)

Trening i optymalizacja Random Forest

In [ ]:
print("Szukanie optymalnych parametrów dla Random Forest za pomocą Optuny (5 prób)...")
best_params_rf, best_score_rf = optimize_rf(X_train, y_train, n_trials=5)

print(f"\nNajlepsze parametry RF: {best_params_rf}")
print(f"Najwyższy wynik CV (F1): {best_score_rf:.4f}")

# Trening ostatecznego modelu
best_model_rf = RandomForestClassifier(**best_params_rf)
best_preds_rf = best_model_rf.fit(X_train, y_train).predict(X_test)

Ewaluacja końcowa - Wyniki i Wykresy

In [ ]:
# ---------------- SVM ----------------
print("="*40)
print("WYNIKI - SUPPORT VECTOR MACHINE (SVM)")
print("="*40)
print(f"Accuracy : {accuracy_score(y_test, best_preds_svm):.4f}")
print(f"F1-score : {f1_score(y_test, best_preds_svm, average='weighted'):.4f}")
print("\nRaport klasyfikacji:")
print(classification_report(y_test, best_preds_svm))

fig, ax = plt.subplots(figsize=(8, 6))
CMD.from_predictions(y_test, best_preds_svm, ax=ax)
plt.title("Macierz Pomyłek - SVM")
plt.show()

# ---------------- Random Forest ----------------
print("\n" + "="*40)
print("WYNIKI - RANDOM FOREST")
print("="*40)
print(f"Accuracy : {accuracy_score(y_test, best_preds_rf):.4f}")
print(f"F1-score : {f1_score(y_test, best_preds_rf, average='weighted'):.4f}")
print("\nRaport klasyfikacji:")
print(classification_report(y_test, best_preds_rf))

fig, ax = plt.subplots(figsize=(8, 6))
CMD.from_predictions(y_test, best_preds_rf, ax=ax)
plt.title("Macierz Pomyłek - Random Forest")
plt.show()

# Podsumowanie oraz analiza
Oba modele dla klasyfikacji wieloklasowej mają umiarkowaną skuteczność z wynikiem (accuracy oraz f1) zbliżonym do 40% na zbiorze testowym (losowe zgadywanie dawałoby ok. 14% accuracy).

Zarówno model SVM, jak i Random Forest najlepiej radzą sobie z rozpoznawaniem neutralnego zabarwienia wypowiedzi. W przypadku klasyfikacji wieloklasowej Random Forest osiąga wyraźnie lepsze wyniki, natomiast SVM sprawdza się lepiej w zadaniu binarnym (RF w tym przypadku może być bardziej podatny na lekkie nierówności w liczebności reprezentajci klas w zbiorze testowym).

Oba modele najczęściej błędnie klasyfikują emocję strachu, generalnie często ją nadając (wysoki recall, niskie precision), myląc ją tym samym z emocjami takimi jak szczęście, smutek oraz zaskoczenie.Dodatkowo klasy „sad” i „happy” są często pomijane, model ma trudności w wykrywaniu tych emocji, co objawia się niskim recall. Wynika to z faktu, że zbiór RAVDESS stanowi trudne zadanie klasyfikacyjne, szczególnie przy uwzględnieniu wszystkich dostępnych klas emocji. Również dla człowieka jednoznaczne określenie stanu emocjonalnego mówcy bywa niejednokrotnie problematyczne.

Zauważalna poprawa wyników pojawia się po ograniczeniu problemu do klasyfikacji binarnej. W takiej konfiguracji oba modele, zarówno w walidacji krzyżowej, jak i na zbiorze testowym, osiągają skuteczność rzędu 90% (zarówno pod względem Accuracy, jak i F1-score).

W celu dalszej poprawy jakości modeli oraz ograniczenia ryzyka przeuczenia (overfittingu), możliwe jest zastosowanie augmentacji danych (np. pitch shifting, time-stretching) lub połączenie zbioru RAVDESS z innym zbiorem danych o podobnej tematyce. Dodatkowo można rozważyć rozszerzenie zestawu obliczanych cech statystycznych o takie miary jak skośność czy kurtoza, co mogłoby dostarczyć modelom dodatkowych informacji o charakterystyce sygnału.